In [1]:
from deepagents import create_deep_agent
from config.llm import zp_llm
from config.tavily_config import tavily_tool_client


/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! openai_api_key is not default parameter.
                openai_api_key was transferred to model_kwargs.
                Please confirm that openai_api_key is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! openai_api_base is not default parameter.
                openai_api_base was transferred to model_kwargs.
                Please confirm that openai_api_base is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
def get_weather(city: str) -> str:
    """get weather for a given city"""
    return f"it's always sunning in {city}"

agent = create_deep_agent(zp_llm, tools=[get_weather], system_prompt="you are a helpful assistant")

agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)


/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! openai_api_key is not default parameter.
                openai_api_key was transferred to model_kwargs.
                Please confirm that openai_api_key is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! openai_api_base is not default parameter.
                openai_api_base was transferred to model_kwargs.
                Please confirm that openai_api_base is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='50d23b1a-2419-43d3-a144-754eed1edaa4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 5675, 'total_tokens': 5727, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2560}}, 'model_provider': 'openai', 'model_name': 'glm-5', 'system_fingerprint': None, 'id': '20260315145543854f3373b1c046e6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf047-7f8f-7500-b007-46f4e98d56a8-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_-7806890585683393799', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5675, 'output_tokens': 52, 'total_tokens': 5727, 'input_token_details': {'cache_read': 2560}, 'output_token_details': {}}),
  ToolMessage(content="it's always sunning i

In [2]:
from typing import Literal

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_tool_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [4]:
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(zp_llm, tools=[internet_search], system_prompt=research_instructions)

In [5]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

# Print the agent's response
print(result["messages"][-1].content)

# What is LangGraph?

**LangGraph** is an open-source framework developed by LangChain for building, deploying, and managing complex AI agent workflows using graph-based architectures. It provides structured control over how LLM-powered applications execute, making it easier to build sophisticated, stateful AI systems.

---

## Core Purpose

LangGraph is designed to **manage the control flow of applications that integrate LLMs**. Unlike simpler chain-based approaches, LangGraph excels when you need:

- **Control over execution flow** — predictable, orchestrated processes
- **State management** — persistent data across workflow steps
- **Complex decision-making** — branching logic and conditional paths
- **Multi-agent coordination** — multiple agents working together

---

## How It Works: Graph-Based Architecture

LangGraph uses a **directed graph structure** with four fundamental building blocks:

| Component | Description |
|-----------|-------------|
| **Graphs** | The overall workf

In [6]:
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call

In [ ]:
call_count = [0]

@wrap_tool_call
def log_tool_calls(request, handler):
    """Intercept and log every tool call - demonstrates cross-cutting concern."""
    call_count[0] += 1
    tool_name = request.name if hasattr(request, "name") else str(request)

    print(f"[Middleware] Tool call #{call_count[0]}: {tool_name}")
    print(f"[Middleware] Arguments: {request.args if hasattr(request, 'args') else 'N/A'}")

    # Execute the tool call
    result = handler(request)

    print(f"[Middleware] Tool call #{call_count[0]} completed")

    return result

agent = create_deep_agent(model=zp_llm, tools=[get_weather], middleware=[log_tool_calls])

In [ ]:
research_subagent = {
    "name": "research-agent",
    "description": "used to research more in depth questions",
    'system_prompt': "you are a great researcher",
    "tools": [internet_search],
    "model": ""
}

subagents = [research_subagent]

agent = create_deep_agent(model=zp_llm, subagents=subagents)


In [ ]:
from langchain_modal import ModalSandbox
import modal

app = modal.App.lookup("your-app")
modal_sandbox = modal.Sandbox.create(app=app)
backend = ModalSandbox(sandbox=modal_sandbox)

agent = create_deep_agent(
    model=zp_llm,
    system_prompt="You are a Python coding assistant with sandbox access.",
    backend=backend,
)